In [0]:
%python
from pyspark.sql.functions import col, lag, datediff, when, count, sum as _sum, round as _round
from pyspark.sql.window import Window

# inpatient stays only — readmission is an inpatient concept
inpatient = (spark.read.table("meridian_dev.silver.encounters")
    .filter(col("encounter_class") == "inpatient")
    .select("encounter_id", "patient_id", "encounter_start", "encounter_stop"))

# the window: per patient, ordered by admission date
w = Window.partitionBy("patient_id").orderBy("encounter_start")

readmissions = (inpatient
    # LAG: the previous inpatient stay's DISCHARGE date for this patient
    .withColumn("prev_discharge", lag("encounter_stop").over(w))
    # days between previous discharge and this admission
    .withColumn("days_since_prev",
        datediff(col("encounter_start"), col("prev_discharge")))
    # flag: readmission if re-admitted within 30 days of prior discharge
    .withColumn("is_readmission",
        when((col("days_since_prev") > 1) & (col("days_since_prev") <= 30), 1).otherwise(0)))

# persist the per-encounter readmission flags
readmissions.write.format("delta").mode("overwrite") \
    .saveAsTable("meridian_dev.gold.readmissions")

# --- compute the overall readmission rate ---
summary = readmissions.agg(
    count("*").alias("total_inpatient_stays"),
    _sum("is_readmission").alias("readmissions"),
    _round(_sum("is_readmission") / count("*") * 100, 2).alias("readmission_rate_pct"))

summary.show()
print("Gold readmissions built.")

In [0]:
SELECT patient_id, encounter_start, prev_discharge, days_since_prev
FROM meridian_dev.gold.readmissions
WHERE is_readmission = 1
ORDER BY days_since_prev
LIMIT 15;